# 02 — Collecte du dataset (SQLite + snowball sampling)

Objectif : construire un pipeline qui collecte des milliers de matchs réels (compositions + résultats) en respectant le rate limiting, avec dédoublonnage et reprise possible après interruption. Ça alimentera la Factorization Machine de la phase 3.

**Mise à jour** : le code de ce notebook a été industrialisé dans le package `lol_assistant/` (voir `README.md`, section "Chantier transversal"). Les explications pédagogiques ci-dessous restent inchangées — seules les cellules de code deviennent de simples imports depuis le package, plutôt que des définitions inline.

**Partie 1 : le schéma SQLite et les fonctions de lecture/écriture.**

## Concept 1 — Schéma SQLite

Deux tables :
- `matches` : le JSON brut de chaque match, `match_id` en clé primaire (donc automatiquement unique — impossible d'insérer deux fois le même match, c'est notre dédoublonnage).
- `explored_players` : les `puuid` dont on a déjà récupéré la liste de matchs — pour ne pas interroger deux fois le même joueur pendant le snowball sampling.

`IF NOT EXISTS` rend la création sûre à ré-exécuter plusieurs fois sans erreur.

**Implémentation** : `lol_assistant/database.py`, fonction `get_connection(db_path)` — elle ouvre la connexion **et** garantit que les tables existent, en un seul appel.

In [1]:
from lol_assistant.database import get_connection

conn = get_connection("../data/matches.db")
print("Connexion prête, tables garanties existantes.")

Connexion prête, tables garanties existantes.


## Concept 2 — Fonctions de lecture/écriture

Quelques principes :
- **Requêtes paramétrées** (`?` dans le SQL, valeurs passées à part) plutôt que d'insérer les valeurs directement dans la chaîne SQL — le réflexe à prendre pour ne jamais être vulnérable à une injection SQL.
- **`INSERT OR IGNORE`** : si la clé primaire existe déjà, SQLite ignore l'insertion silencieusement au lieu de lever une erreur — pratique pour notre dédoublonnage.
- **`json.dumps`** : une table SQL ne stocke pas nativement un dict Python — on le sérialise en texte JSON pour le stocker dans la colonne `raw_json`.

**Implémentation** : `lol_assistant/database.py` — `match_already_collected`, `save_match`, `player_already_explored`, `mark_player_explored`. Différence avec la version d'origine du notebook : chaque fonction prend maintenant `conn` en paramètre explicite, plutôt que de dépendre d'un `cursor` global — pas de risque d'état partagé entre fonctions.

In [2]:
from lol_assistant.database import (
    match_already_collected,
    save_match,
    player_already_explored,
    mark_player_explored,
)

print("Fonctions importées.")

Fonctions importées.


## Petit test de sanité (sans appel API)

On vérifie que les fonctions marchent avec des données factices, avant de les brancher sur de vraies données Riot.

In [3]:
fake_id = "TEST_1234"

print("Avant insertion :", match_already_collected(conn, fake_id))  # attendu: False

save_match(conn, fake_id, "euw1", {"hello": "world"})

print("Après insertion :", match_already_collected(conn, fake_id))  # attendu: True

# on nettoie tout de suite, c'était juste un test
conn.execute("DELETE FROM matches WHERE match_id = ?", (fake_id,))
conn.commit()
print("Après nettoyage :", match_already_collected(conn, fake_id))  # attendu: False

Avant insertion : False
Après insertion : True
Après nettoyage : False


---

**Checkpoint** : exécute ces cellules et confirme que le test de sanité affiche bien `False` / `True` / `False`. Ensuite on enchaîne sur le rate limiter proactif et la boucle de snowball sampling.

## Concept 3 — Rate limiter proactif

Riot impose deux limites simultanées pour une clé de dev : 20 requêtes/seconde **et** 100 requêtes/2 minutes. En usage soutenu, la limite des 2 minutes est de loin la plus contraignante (50 req/min en moyenne, très en dessous de 20/s) — donc respecter uniquement celle-ci suffit en pratique.

Le principe d'une **fenêtre glissante** : on garde en mémoire les instants (timestamps) des derniers appels. Avant chaque nouvel appel, on retire de cette mémoire les timestamps plus vieux que 120 secondes, puis on regarde combien il en reste. S'il y en a déjà 100, on attend le temps nécessaire pour que le plus ancien sorte de la fenêtre.

**Implémentation** : `lol_assistant/rate_limiter.py`, classe `RateLimiter`.

In [4]:
from lol_assistant.rate_limiter import RateLimiter
import time

print("RateLimiter importé.")

RateLimiter importé.


**Test du mécanisme** : avec de petites valeurs (3 appels max par tranche de 5 secondes), pour voir l'attente se déclencher en quelques secondes plutôt que d'attendre 2 minutes avec les vraies limites Riot.

In [5]:
test_limiter = RateLimiter(max_calls=3, period_seconds=5)

for i in range(5):
    test_limiter.wait_if_needed()
    print(f"Appel {i + 1} effectué à t={time.time():.1f}")

Appel 1 effectué à t=1789475551.3
Appel 2 effectué à t=1789475551.3
Appel 3 effectué à t=1789475551.3


Appel 4 effectué à t=1789475556.3
Appel 5 effectué à t=1789475556.3


**Ce que tu devrais observer** : les appels 1, 2, 3 s'enchaînent quasi instantanément (moins de 3 appels dans la fenêtre de 5s, pas d'attente). Au 4ème appel, l'appel est retardé jusqu'à ce que le 1er appel sorte de la fenêtre de 5 secondes.

## Concept 4 — Le client Riot

Ce qui était des variables globales (`api_key`, `headers`, `region`) et une fonction `riot_get` devient une classe `RiotClient` : elle encapsule l'authentification, la région, et une instance de `RateLimiter`, avec trois méthodes de haut niveau — `get_puuid`, `get_match_ids`, `get_match_detail` — qui construisent elles-mêmes les bonnes URLs.

Différence de comportement à noter : chaque méthode appelle `response.raise_for_status()` — au lieu de renvoyer un objet `response` qu'il faut vérifier manuellement (`if response.status_code != 200`), elle **lève une exception** (`requests.HTTPError`) en cas d'erreur. Le code appelant utilisera donc `try`/`except` plutôt qu'un `if`.

**Implémentation** : `lol_assistant/riot_client.py`, classe `RiotClient`.

In [6]:
from lol_assistant.riot_client import RiotClient

client = RiotClient(region="europe")
print("Client Riot prêt.")

Client Riot prêt.


---

**Checkpoint** : exécute ces nouvelles cellules (le test du rate limiter, puis la création du `RiotClient`). Il n'y a pas encore de vrai appel à l'API Riot à ce stade — on vérifie juste que tout s'exécute sans erreur. Ensuite, dernière pièce du puzzle de la phase 2 : la boucle de crawl (snowball sampling), déjà entièrement implémentée dans `lol_assistant/crawler.py`.

## Concept 5 — La boucle de crawl (snowball sampling)

C'est un parcours en largeur (BFS) sur le graphe des joueurs :
1. On part d'un `puuid` de départ, placé dans une **file d'attente**.
2. Tant que la file n'est pas vide et qu'on n'a pas atteint le nombre de matchs voulu : on prend le prochain joueur de la file, on récupère sa liste de matchs récents.
3. Pour chaque match pas encore en base : on télécharge le détail, on le sauvegarde, et on ajoute ses 10 participants à la file — nouvelles graines, obtenues "gratuitement" sans appel API dédié.
4. On marque le joueur comme exploré pour ne jamais retraiter sa liste de matchs deux fois.

**Implémentation** : `lol_assistant/crawler.py`, fonction `crawl_matches(riot_client, conn, seed_puuid, target_count, matches_per_player=10)` — elle orchestre `RiotClient` et les fonctions de `database.py` plutôt que de dépendre de fonctions/variables globales.

In [7]:
from lol_assistant.crawler import crawl_matches

print("crawl_matches importé.")

crawl_matches importé.


Petite précision : contrairement à `info.participants` (utilisé en phase 1 pour extraire champion/équipe/rôle), le crawler utilise `metadata.participants` — directement une simple liste des 10 `puuid` de la partie, exactement ce qu'il nous faut pour alimenter la file.

## Récupérer le `puuid` de départ

Via `client.get_puuid(game_name, tag_line)` — encapsule l'appel Account-V1 déjà expliqué en phase 1.

In [8]:
seed_puuid = client.get_puuid("Spear Shot", "1111")
print("Seed puuid:", seed_puuid)

Seed puuid: q2DwelIuM6EaCKUbCqjM8l-LBxmoU53eKPvnlb3DhEj75ht56ab9VJ_F4SSJzKnLG64useI2M1BbyQ


## Premier test à petite échelle

Avant de viser des milliers de matchs (ce qui prendra du temps, vu le rate limiting), on lance une collecte de seulement **20 matchs** pour valider que tout le pipeline fonctionne correctement de bout en bout.

In [9]:
crawl_matches(client, conn, seed_puuid, target_count=20)

Terminé : 0 matchs collectés, 0 joueurs encore en attente dans la file.


In [10]:
total_matches = conn.execute("SELECT COUNT(*) FROM matches").fetchone()[0]
print("Total matchs en base:", total_matches)

total_players = conn.execute("SELECT COUNT(*) FROM explored_players").fetchone()[0]
print("Total joueurs explorés:", total_players)

Total matchs en base: 20
Total joueurs explorés: 3


---

**Checkpoint final de la phase 2** : exécute tout, du seed `puuid` jusqu'à la vérification. `data/matches.db` contient déjà des matchs d'une exécution précédente — le compteur total sera donc supérieur à 20 (20 nouveaux matchs s'ajoutent aux existants, grâce au dédoublonnage).

Pipeline complet, testé, et maintenant industrialisé dans `lol_assistant/`. Pour la vraie collecte à grande échelle, il suffira de relancer `crawl_matches` avec un `target_count` bien plus grand — en gardant en tête le renouvellement de la clé API (expiration 24h).

## Bonus — Visualiser le contenu de la base avec pandas

`pandas` sait exécuter une requête SQL directement sur une connexion `sqlite3` et renvoyer le résultat sous forme de tableau (`DataFrame`), avec un affichage propre dans Jupyter — pratique pour explorer visuellement sans écrire de boucle Python à la main.

In [11]:
import pandas as pd

df = pd.read_sql_query("SELECT match_id, region FROM matches", conn)
df

,match_id,region
0,EUW1_7981560267,europe
1,EUW1_7981526567,europe
2,EUW1_7979271688,europe
3,EUW1_7978928820,europe
4,EUW1_7978885105,europe
5,EUW1_7978830786,europe
6,EUW1_7978488773,europe
7,EUW1_7978434516,europe
8,EUW1_7978378760,europe
9,EUW1_7977766971,europe


On n'affiche pas `raw_json` ici : c'est un gros bloc de texte, illisible dans un tableau. On ira le décortiquer (`json.loads` + extraction champion/équipe/rôle) en phase 3, quand on construira le vrai dataset d'entraînement.